# JalVaani v4.1 — groundwater forecasting (t+7 / t+14 / t+30), improvement run + simple results

**Before you start:** Runtime → Change runtime type → **T4 GPU** → Save. Then Runtime → Run all.

**Upload in Step 1.** All of these are in `Models/TFT/` on your laptop:

| file | what it is |
|---|---|
| `tft_model_v3.py` | the TFT network (unchanged) |
| `tft_v4.py` | the pipeline, **v4.5 or newer** |
| `master_dataset_v4target.csv` | the dataset (25 districts, 2023-01-01 → 2026-06-04) |
| `RESULTS_GUIDE.md` | the guide; it is copied into the results zip |
| `tft_metrics_per_district.csv` | *optional* — the OLD v1 file from `Latest results/`, for a side-by-side table |

**What it does (about 1.5 h):**
1. **Screening** on dev folds 1–4 (3 seeds). It tries a stronger LightGBM, "fresher" training that also learns the last 3 months, stride-1 sampling and a wider network. The pick is made by a rule fixed in advance.
2. **Tuning sweeps** on the chosen set-up: per-district stacking strength and interval adaptivity. These are free, because no retraining is needed.
3. **Final run** on all 6 folds with 10 seeds. Folds 5–6 are reported separately as the locked test.
4. **Simple results** in the old v1 style, for **all 24 districts**, for both the **Ensemble** and the **TFT alone**: tables, per-district plots, coverage heatmap, training curves and VSN importance.
5. Downloads `tft_v4_outputs.zip`. Read `RESULTS_GUIDE.md` inside it.

In [ ]:
# ── STEP 1: upload the files listed above (from your laptop) ─────────────────
from google.colab import files
uploaded = files.upload()
print('Uploaded:', list(uploaded.keys()))

In [ ]:
!pip install -q lightgbm

In [ ]:
# ── STEP 2: setup ────────────────────────────────────────────────────────────
import sys, time, shutil, zipfile, os
from pathlib import Path
import numpy as np, pandas as pd, torch
from IPython.display import Image, display, Markdown
sys.path.insert(0, '/content')
import tft_v4 as v4

assert torch.cuda.is_available(), 'No GPU: Runtime > Change runtime type > T4 GPU, then Run all again'
assert tuple(map(int, v4.PIPELINE_VERSION.split('.'))) >= (4, 5), f'old tft_v4.py ({v4.PIPELINE_VERSION}) - upload the new one'
print('GPU:', torch.cuda.get_device_name(0), '| pipeline', v4.PIPELINE_VERSION)

DATA = '/content/master_dataset_v4target.csv'
OUT  = Path('/content/outputs/v4')
OUT.mkdir(parents=True, exist_ok=True)
assert Path(DATA).exists(), 'master_dataset_v4target.csv was not uploaded'
T0 = time.time()
def elapsed(): return (time.time() - T0) / 60
pd.set_option('display.width', 220); pd.set_option('display.max_columns', 40)

In [ ]:
# ── STEP 3: settings ─────────────────────────────────────────────────────────
QUICK       = False          # True = skip screening + sweeps; run the most likely best set-up directly (~45 min)
DEV_SEEDS   = [42, 1337, 2026]
FINAL_SEEDS = DEV_SEEDS + [7, 99, 123, 555, 777, 2024, 31337]   # 10; cut to 7 if the run is behind schedule

BASE  = dict(target_mode='snaive', calib_mode='v4', combine='stack')        # the v4.0 set-up
TAB2  = dict(tab_features='v2', tab_stride=1, tab_fresh=True,               # stronger LightGBM
             gbm_n_estimators=400, gbm_lr=0.03, gbm_min_child=100)
NET_A = dict(refit=True)                                                    # networks: pass 1 + refit pass
NET_B = dict(refit=True, train_stride=1, epoch_frac=1/7)                    # + stride-1 sampling
NET_C = dict(refit=True, d_model=48)                                        # + wider network

# Order = simplicity order for ties. Candidates sharing NET_x reuse the same trained networks;
# candidates sharing TAB2 reuse the same LightGBM - only 3 network trainings in total.
CANDIDATES = {
    'ref':            {**BASE, **NET_A, 'use_refit': False},          # = v4.0 (regression check)
    'tab2':           {**BASE, **NET_A, **TAB2, 'use_refit': False},  # + stronger LightGBM
    'refit':          {**BASE, **NET_A, 'use_refit': True},           # + fresher networks
    'tab2_refit':     {**BASE, **NET_A, **TAB2, 'use_refit': True},
    's1_tab2':        {**BASE, **NET_B, **TAB2, 'use_refit': False},
    's1_tab2_refit':  {**BASE, **NET_B, **TAB2, 'use_refit': True},
    'd48_tab2_refit': {**BASE, **NET_C, **TAB2, 'use_refit': True},
}
QUICK_PICK = 'tab2_refit'

def make(tag, mode='dev', seeds=DEV_SEEDS, **kw):
    cfg = v4.Config(tag=tag, mode=mode, data_path=DATA, out_root=str(OUT), seeds=list(seeds))
    for k, v in kw.items():
        setattr(cfg, k, v)
    return cfg

def brief(tag, S):
    rows = []
    for name, sm in S.items():
        h, c = sm['head'], sm['cov']
        rows.append({'run': tag, 'model': name,
                     **{f'vs no-change {k}': h.loc[k, 'rMAE_P'] for k in v4.HORDER},
                     **{f'vs seasonal {k}': h.loc[k, 'rMAE_SN'] for k in v4.HORDER},
                     **{f'cov {k}': c.loc[k, 'Cov (%)'] for k in v4.HORDER},
                     **{f'below {k}': c.loc[k, 'Below q10 (%)'] for k in v4.HORDER},
                     **{f'above {k}': c.loc[k, 'Above q90 (%)'] for k in v4.HORDER},
                     'interval score': c['Interval score (m)'].mean(),
                     'mean vs seasonal': h['rMAE_SN'].mean()})
    return pd.DataFrame(rows)

def run_dev(tag, **kw):
    res = v4.run(make(tag, **kw), verbose=False)
    return v4.report(res, verbose=False, save_predictions=False)

## Screening (dev folds 1–4)
**Pick rule, fixed before the run:** lowest mean Ensemble error ÷ seasonal-naive error over t+7, t+14 and t+30. A simpler candidate within 0.002 of the best wins.

In [ ]:
SCREEN = []
if not QUICK:
    for tag, kw in CANDIDATES.items():
        print(f'######## dev: {tag}   [{elapsed():.0f} min elapsed] ########', flush=True)
        SCREEN.append(brief(tag, run_dev('dev_' + tag, **kw)))
        display(SCREEN[-1].round(3))
    SCREEN = pd.concat(SCREEN, ignore_index=True)
    SCREEN.to_csv(OUT / 'dev_screening_summary.csv', index=False)
    ens = SCREEN[SCREEN.model == 'Ensemble'].set_index('run')['mean vs seasonal']
    BEST = next(c for c in CANDIDATES if c in ens.index and ens[c] <= ens.min() + 0.002)
    ref = SCREEN[(SCREEN.run == 'ref') & (SCREEN.model == 'Ensemble')].iloc[0]
    print('\nRegression check - ref Ensemble vs seasonal (v4.0 dev was 0.973 / 0.966 / 0.951):',
          [round(ref[f'vs seasonal {h}'], 3) for h in v4.HORDER])
    display(SCREEN[SCREEN.model == 'Ensemble'].sort_values('mean vs seasonal')
            [['run', 'mean vs seasonal'] + [f'vs no-change {h}' for h in v4.HORDER] + [f'cov {h}' for h in v4.HORDER]].round(3))
else:
    BEST = QUICK_PICK
print(f'Chosen set-up: {BEST}   [{elapsed():.0f} min elapsed]')

## Tuning sweeps on the chosen set-up (no retraining)
* **Stacking strength** `stack_prior_n` ∈ {100, 250, 500, 1000}. A smaller value lets each district lean more on its own best component. Pick: lowest mean Ensemble vs seasonal; 500 is kept unless another value beats it by more than 0.001.
* **Interval adaptivity** `aci_gamma` ∈ {0.005, 0.01, 0.02, 0.05}. Pick: both tails within 5–15% at every horizon, then closest to 80% coverage. Ties within 0.5 points keep 0.01.

In [ ]:
best_kw = dict(CANDIDATES[BEST])
N, G = 500, 0.01
SWEEP = []
if not QUICK:
    base_row = SCREEN[(SCREEN.run == BEST) & (SCREEN.model == 'Ensemble')].iloc[0]
    spn = {500: base_row}
    for n in (100, 250, 1000):
        print(f'sweep stack_prior_n={n}   [{elapsed():.0f} min]', flush=True)
        b = brief(f'spn{n}', run_dev(f'sweep_spn{n}', **{**best_kw, 'stack_prior_n': n}))
        SWEEP.append(b); spn[n] = b[b.model == 'Ensemble'].iloc[0]
    best_n = min(spn, key=lambda n: spn[n]['mean vs seasonal'])
    N = best_n if spn[best_n]['mean vs seasonal'] < spn[500]['mean vs seasonal'] - 0.001 else 500

    gam = {0.01: spn[N]}
    for g in (0.005, 0.02, 0.05):
        print(f'sweep aci_gamma={g}   [{elapsed():.0f} min]', flush=True)
        b = brief(f'aci{g}', run_dev(f'sweep_aci{g}', **{**best_kw, 'stack_prior_n': N, 'aci_gamma': g}))
        SWEEP.append(b); gam[g] = b[b.model == 'Ensemble'].iloc[0]
    def cov_score(r): return np.mean([abs(r[f'cov {h}'] - 80) for h in v4.HORDER])
    def feasible(r): return all(5 <= r[f'{s} {h}'] <= 15 for s in ('below', 'above') for h in v4.HORDER)
    pool_g = [g for g in gam if feasible(gam[g])] or list(gam)
    g_best = min(pool_g, key=lambda g: cov_score(gam[g]))
    G = g_best if cov_score(gam[g_best]) < cov_score(gam[0.01]) - 0.5 or 0.01 not in pool_g else 0.01
    SWEEP = pd.concat(SWEEP, ignore_index=True)
    SWEEP.to_csv(OUT / 'post_sweep_summary.csv', index=False)
    display(SWEEP[SWEEP.model == 'Ensemble'].round(3))
FINAL_KW = {**best_kw, 'stack_prior_n': N, 'aci_gamma': G}
print(f'Final set-up: {BEST}  stack_prior_n={N}  aci_gamma={G}   [{elapsed():.0f} min elapsed]')

## Final run — all 6 folds, 10 seeds
All six folds are trained, because folds 1–4 feed the calibration pool of later folds. Results are reported twice:
- all 6 folds;
- folds 5–6 only: the locked test, which nothing in v4.1 was tuned on.

In [ ]:
n_seeds = 10 if (elapsed() <= 50 or QUICK) else 7
FINAL_TAG = 'FINAL_' + BEST
print(f'Final run with {n_seeds} seeds   [{elapsed():.0f} min elapsed]')
res = v4.run(make(FINAL_TAG, mode='all', seeds=FINAL_SEEDS[:n_seeds], **FINAL_KW), verbose=True)
print('\n\n================  ALL 6 FOLDS  ================')
S_all = v4.report(res, verbose=True)

In [ ]:
print('================  LOCKED TEST: folds 5-6 (Oct 2025 -> Jun 2026)  ================')
locked = v4.subset_folds(res, list(v4.LOCKED_FOLDS), FINAL_TAG + '_locked')
S_locked = v4.report(locked, verbose=True, save_predictions=False)

QUOTE = v4.results_to_quote(S_locked, S_all)
QUOTE.to_csv(OUT / 'RESULTS_TO_QUOTE.csv', index=False)

# v4.0 headline (Colab run of 2026-10-03) for comparison
V4 = pd.DataFrame([
    (t, m, h, mae, p, s, c) for (t, m), vals in {
        ('locked folds 5-6', 'Ensemble'): [(0.3234, .8645, .9666, 80.6), (0.4192, .7876, .9521, 81.5), (0.5928, .6640, .9446, 81.2)],
        ('locked folds 5-6', 'TFT'):      [(0.3403, .9097, 1.0171, 74.0), (0.4438, .8338, 1.0080, 70.1), (0.6225, .6973, .9919, 66.8)],
        ('all 6 folds', 'Ensemble'):      [(0.4327, .9108, .9748, 80.8), (0.6099, .8503, .9720, 80.3), (0.9069, .7502, .9582, 77.7)],
        ('all 6 folds', 'TFT'):           [(0.4452, .9371, 1.0029, 74.1), (0.6286, .8765, 1.0019, 69.4), (0.9339, .7725, .9868, 65.0)],
    }.items() for h, (mae, p, s, c) in zip(v4.HORDER, vals)],
    columns=['test set', 'model', 'horizon', 'v4.0 MAE (m)', 'v4.0 vs no-change', 'v4.0 vs seasonal', 'v4.0 coverage'])
CMP = V4.merge(QUOTE[['test set', 'model', 'horizon', 'MAE (m)', 'rMAE vs persistence', 'rMAE vs seasonal',
                      'coverage 80% (%)']].rename(columns={'MAE (m)': 'v4.1 MAE (m)', 'rMAE vs persistence': 'v4.1 vs no-change',
                                                          'rMAE vs seasonal': 'v4.1 vs seasonal', 'coverage 80% (%)': 'v4.1 coverage'}),
               on=['test set', 'model', 'horizon'])
CMP['change vs no-change'] = (CMP['v4.1 vs no-change'] - CMP['v4.0 vs no-change']).round(3)
CMP.to_csv(OUT / 'v4_vs_v4_1.csv', index=False)
display(Markdown('### v4.0 → v4.1 (lower ratios are better)')); display(CMP.round(3))
display(Markdown('### Results to quote')); display(QUOTE.round(3))

## Simple results (v1 style): all 24 districts, Ensemble and TFT

In [ ]:
SIMPLE = OUT / 'simple'
for model in ('Ensemble', 'TFT'):
    v4.export_simple(res, model, SIMPLE / model.lower())
v4.export_shared(res, SIMPLE)
if Path('/content/tft_metrics_per_district.csv').exists():
    v4.compare_v1('/content/tft_metrics_per_district.csv', SIMPLE)
    print('v1 comparison written (NOT like-for-like - see its caveat column)')
if Path('/content/RESULTS_GUIDE.md').exists():
    shutil.copy('/content/RESULTS_GUIDE.md', OUT / 'RESULTS_GUIDE.md')
else:
    print('RESULTS_GUIDE.md was not uploaded - the zip will not contain the guide')

lines = [f'# JalVaani v4.1 run summary', '',
         f'- pipeline {v4.PIPELINE_VERSION}, data {Path(DATA).name} (md5 {v4.file_md5(DATA)[:8]})',
         f'- chosen set-up: **{BEST}** = {CANDIDATES[BEST]}',
         f'- stacking strength stack_prior_n = {N}, interval adaptivity aci_gamma = {G}',
         f'- final seeds: {FINAL_SEEDS[:n_seeds]}', f'- total run time: {elapsed():.0f} min',
         '- selection used dev folds 1-4 only; folds 5-6 are the locked test (viewed once before, in v4.0)', '']
if len(SCREEN):
    lines += ['## Screening (dev folds 1-4)', '```',
              SCREEN[SCREEN.model == 'Ensemble'][['run', 'mean vs seasonal'] + [f'vs no-change {h}' for h in v4.HORDER]]
              .round(3).to_string(index=False), '```', '']
lines += ['## v4.0 -> v4.1', '```', CMP.round(3).to_string(index=False), '```']
(OUT / 'RUN_SUMMARY.md').write_text('\n'.join(lines), encoding='utf-8')
print(f'done   [{elapsed():.0f} min elapsed]')

In [ ]:
# ── the key results, shown here ──────────────────────────────────────────────
for model in ('ensemble', 'tft'):
    display(Markdown(f'### {model.upper()} — headline (v1-style)'))
    display(pd.read_csv(SIMPLE / model / f'{model}_metrics_headline.csv').round(3))
display(Markdown('### Ensemble — MASE per district (error ÷ no-change error; below 1 is better)'))
display(pd.read_csv(SIMPLE / 'ensemble' / 'ensemble_mase_by_district.csv').round(3))
for f in ['ensemble/ensemble_per_district_mase_t30.png', 'ensemble/ensemble_per_district_mape_t30.png',
          'ensemble/ensemble_coverage_heatmap.png', 'ensemble/ensemble_forecast_grid_t30.png',
          'tft/tft_per_district_mase_t30.png', 'tft_training_curves.png', 'tft_vsn_importance.png',
          'ensemble/quantile/quantile_nagpur.png', 'ensemble/quantile/quantile_washim.png']:
    p = SIMPLE / f
    if p.exists():
        display(Markdown(f'**{f}**')); display(Image(filename=str(p), width=1100))

In [ ]:
# ── zip everything except the model caches, and download ────────────────────
zpath = '/content/tft_v4_outputs.zip'
with zipfile.ZipFile(zpath, 'w', zipfile.ZIP_DEFLATED) as z:
    for p in OUT.rglob('*'):
        if p.is_file() and '_cache' not in p.parts:
            z.write(p, p.relative_to(OUT))
print(f'{zpath}: {os.path.getsize(zpath)/1e6:.1f} MB')
files.download(zpath)